# 🎙️ Projeto Aigis — Treinamento e Destilação da Voz (Piper PT-BR 24kHz)

Este notebook executa o treinamento completo da persona **Aigis** a partir do dataset **Padrão Ouro Sweetspot (PT-BR)**.

### ⚡ Benefícios no Google Colab:
1. **Ambiente Isolado com Python 3.10**: Resolve 100% dos erros de compatibilidade do Piper/Cython.
2. **Persistência no Google Drive (2 TB)**: Salva checkpoints continuamente na pasta `Aigis_TTS_Model` do seu Drive.
3. **Zero perda**: Se o Colab reiniciar, basta rodar novamente que ele retoma do último checkpoint salvo.

---
### 📌 Instruções Simples:
1. No menu superior, vá em **Ambiente de execução -> Alterar tipo de ambiente de execução** e escolha **GPU (T4 ou A100)**.
2. Certifique-se de que o arquivo `dataset.zip` (45 MB) está na pasta `Aigis_TTS_Model` do seu Google Drive.
3. Execute a **Célula 1** (para autorizar o Google Drive).
4. Execute a **Célula 2** para rodar o treinamento completo!

In [ ]:
#@title 1. Conectar ao Google Drive e Verificar GPU
import os
from pathlib import Path
from IPython.display import display, Javascript

print("[*] Conectando ao Google Drive...")
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = Path('/content/drive/MyDrive/Aigis_TTS_Model')
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
print(f"✅ Google Drive conectado! Pasta de persistência: {DRIVE_DIR}")

# Auto-keepalive para manter a sessão ativa
display(Javascript('''
  function ClickConnect(){
    console.log("[Aigis Keepalive] Mantendo sessão ativa...");
    document.querySelector("colab-connect-button")?.shadowRoot?.querySelector("#connect")?.click();
  }
  setInterval(ClickConnect, 60000);
'''))

!nvidia-smi

In [ ]:
#@title 2. Executar Treinamento Automatizado (Ambiente Python 3.10 Isolado)
%cd /content

# Cria o script mestre de treinamento com todas as correções de ambiente
script_content = '''
import os, sys, shutil, subprocess, zipfile, py_compile
from pathlib import Path

os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"
ENV_DIR = "/content/piper_env"
ENV_PY = f"{ENV_DIR}/bin/python3"
ENV_PIP = f"{ENV_DIR}/bin/pip"
DRIVE_DIR = Path("/content/drive/MyDrive/Aigis_TTS_Model")
BASE_CKPT = "pt_BR-faber-medium.ckpt"

def run(cmd):
    print(f"\n[EXEC] {cmd}")
    res = subprocess.run(cmd, shell=True)
    if res.returncode != 0:
        raise RuntimeError(f"Erro no comando: {cmd}")

print("="*60)
print("🚀 PREPARANDO AMBIENTE PYTHON 3.10 ISOLADO PARA O PIPER")
print("="*60)

# 1. Instala Python 3.10 e espeak-ng
run("apt-get update -qq && apt-get install -y -qq espeak-ng libespeak-ng-dev python3.10 python3.10-venv python3.10-dev")

# 2. Cria venv com Python 3.10
if not os.path.exists(ENV_DIR):
    run(f"python3.10 -m venv {ENV_DIR}")

# 3. Instala dependências compatíveis no venv
run(f"{ENV_PIP} install --upgrade pip==24.0")
run(f"{ENV_PIP} install torch torchaudio --index-url https://download.pytorch.org/whl/cu121")
wheel_url = "https://github.com/rhasspy/piper-phonemize/releases/download/v1.1.0/piper_phonemize-1.1.0-cp310-cp310-manylinux_2_28_x86_64.whl"
run(f"{ENV_PIP} install '{wheel_url}'")
run(f'{ENV_PIP} install "numpy<2.0" piper-tts onnx onnxruntime pytorch-lightning==1.9.5 torchmetrics==0.11.4 tensorboard cython librosa==0.9.2')

# 4. Clona piper e compila monotonic_align
if not os.path.exists("piper"):
    run("git clone https://github.com/rhasspy/piper.git")

align_dir = "piper/src/python/piper_train/vits/monotonic_align"
run(f"cd {align_dir} && mkdir -p monotonic_align && {ENV_DIR}/bin/cythonize -i core.pyx")
run(f"cd {align_dir} && cp -f core*.so monotonic_align/ 2>/dev/null || true")
run(f"{ENV_PIP} install --no-deps -e piper/src/python")

# 5. Prepara dataset
if not os.path.exists("dataset.zip"):
    if (DRIVE_DIR / "dataset.zip").exists():
        print(f"[*] Copiando dataset.zip do Google Drive ({DRIVE_DIR}/dataset.zip)...")
        shutil.copy(str(DRIVE_DIR / "dataset.zip"), "dataset.zip")
    else:
        raise FileNotFoundError("dataset.zip não encontrado! Por favor coloque o arquivo na pasta Aigis_TTS_Model do Drive.")

if not os.path.exists("dataset/metadata.csv"):
    print("[*] Descompactando dataset.zip...")
    with zipfile.ZipFile("dataset.zip", "r") as z:
        z.extractall(".")
    nested = Path("dataset/dataset")
    if nested.exists() and (nested / "metadata.csv").exists():
        for it in nested.iterdir():
            dest = Path("dataset") / it.name
            if not dest.exists(): shutil.move(str(it), str(dest))
        shutil.rmtree(nested, ignore_errors=True)

# 6. Baixa checkpoint base Faber Medium
if not os.path.exists(BASE_CKPT):
    print("[*] Baixando checkpoint base Faber Medium...")
    ckpt_url = "https://huggingface.co/datasets/rhasspy/piper-checkpoints/resolve/main/pt/pt_BR/faber/medium/epoch%3D6159-step%3D1230728.ckpt"
    run(f"wget -q -O {BASE_CKPT} '{ckpt_url}'")

# 7. Pré-processamento
if not os.path.exists("training_output/dataset.jsonl"):
    print("[*] Pré-processando dataset para LJSpeech...")
    run(f"{ENV_PY} -m piper_train.preprocess --language pt-br --input-dir dataset --output-dir training_output --dataset-format ljspeech --single-speaker --sample-rate 22050")

# 8. Treinamento com sincronização contínua
print("="*60)
print("🏋️ INICIANDO TREINAMENTO NO COLAB COM GPU")
print("="*60)

# Identifica último checkpoint se houver
drive_ckpts = list(DRIVE_DIR.glob("*.ckpt"))
resume_ckpt = max(drive_ckpts, key=os.path.getmtime) if drive_ckpts else Path(BASE_CKPT)
print(f"[*] Checkpoint inicial: {resume_ckpt.name}")

train_cmd = (
    f"{ENV_PY} -m piper_train "
    f"--dataset-dir training_output "
    f"--accelerator gpu "
    f"--devices 1 "
    f"--batch-size 32 "
    f"--checkpoint-epochs 25 "
    f"--precision 32 "
    f"--quality medium "
    f"--max_epochs 20000 "
    f"--resume_from_checkpoint {resume_ckpt}"
)
run(train_cmd)
'''

with open('/content/run_piper_colab.py', 'w', encoding='utf-8') as f:
    f.write(script_content)

# Executa o script mestre com saída em tempo real
!python3 /content/run_piper_colab.py

In [ ]:
#@title 3. Exportar Modelo Final ONNX e Tocar Áudio
%cd /content
import os, shutil, glob
from pathlib import Path
from IPython.display import Audio, display

ENV_DIR = "/content/piper_env"
ENV_PY = f"{ENV_DIR}/bin/python3"
DRIVE_DIR = Path("/content/drive/MyDrive/Aigis_TTS_Model")

ckpts = list(Path("training_output/lightning_logs").glob("**/*.ckpt")) + list(DRIVE_DIR.glob("*.ckpt"))
if not ckpts:
    raise FileNotFoundError("Nenhum checkpoint encontrado!")

best_ckpt = max(ckpts, key=os.path.getmtime)
print(f"[*] Exportando ONNX a partir de: {best_ckpt}")

!{ENV_PY} -m piper_train.export_onnx "{best_ckpt}" /content/aigis_tts_ptbr.onnx
!cp /content/training_output/config.json /content/aigis_tts_ptbr.onnx.json

# Salva no Google Drive
!cp /content/aigis_tts_ptbr.onnx /content/drive/MyDrive/Aigis_TTS_Model/aigis_tts_ptbr.onnx
!cp /content/aigis_tts_ptbr.onnx.json /content/drive/MyDrive/Aigis_TTS_Model/aigis_tts_ptbr.onnx.json
print(f"✅ Modelo salvo com sucesso em: {DRIVE_DIR}")

# Teste com áudio direto
!echo "Olá, mestre! Como posso ajudar você hoje?" | {ENV_DIR}/bin/piper \
  --model /content/aigis_tts_ptbr.onnx \
  --config /content/aigis_tts_ptbr.onnx.json \
  --output_file /content/teste_aigis_final.wav

if os.path.exists('/content/teste_aigis_final.wav') and os.path.getsize('/content/teste_aigis_final.wav') > 1000:
    display(Audio('/content/teste_aigis_final.wav', autoplay=True))
else:
    print('❌ Arquivo de áudio não foi gerado. Verifique os logs de exportação acima.')